In [1]:
import sys
import os
import logging
import gc
import time
import torch
import warnings
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gliner import GLiNER
from gliner.data_processing.collator import DataCollator
from gliner.training import Trainer, TrainingArguments
from transformers import TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType,PeftModel

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
sys.path

/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python311.zip',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11/lib-dynload',
 '',
 '/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages',
 '/tmp/tmpe_6gp61o']

In [2]:
src_path=os.path.join(os.path.dirname(os.getcwd()),'src')
sys.path.append(src_path)
sys.path


['/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python311.zip',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11/lib-dynload',
 '',
 '/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages',
 '/tmp/tmpe_6gp61o',
 '/opt/app/notebooks/abhishek/active_gliner/src']

In [3]:
from config.settings import Settings
settings = Settings()

print(f"Settings cache_dir: {settings.cache_dir}")
print(f"Cache absolute path: {settings.cache_dir.resolve()}")
print(f"Does cache dir contain 'notebooks': {'notebooks' in str(settings.cache_dir)}")

Settings cache_dir: /opt/app/notebooks/abhishek/active_gliner/cache
Cache absolute path: /opt/app/notebooks/abhishek/active_gliner/cache
Does cache dir contain 'notebooks': True


In [4]:
settings

Settings(seed=42, batch_size=8, model=knowledgator/modern-gliner-bi-large-v1.0)

In [5]:
print("=== Integration Test ===")
from utils.logging import setup_logging
from utils.reproducibility import set_all_seeds
from utils.device import setup_device

# Complete setup like your original code
settings = Settings()
settings.setup()  # Apply environment and create directories

logger = setup_logging(log_dir=str(settings.logs_dir))
set_all_seeds(seed=settings.global_seed, logger=logger)
device = setup_device(logger=logger)

logger.info("All modules integrated successfully!")
print(f"Final setup: seed={settings.global_seed}, device={device}, batch_size={settings.batch_size}")



INFO:ActiveLearning:================================================================================
INFO:ActiveLearning:ACTIVE LEARNING PIPELINE WITH PROPER TRAIN/TEST SEPARATION
INFO:ActiveLearning:================================================================================
INFO:ActiveLearning:Log file: /opt/app/notebooks/abhishek/active_gliner/logs/active_learning_20250910_190512.log
INFO:ActiveLearning:Setting all seeds to 42 for reproducibility...
INFO:ActiveLearning:Using device: cuda
INFO:ActiveLearning:CUDA version: 12.8
INFO:ActiveLearning:Number of GPUs visible: 1
INFO:ActiveLearning:Current GPU: 0
INFO:ActiveLearning:GPU Name: NVIDIA GeForce RTX 3090
INFO:ActiveLearning:GPU Memory: 23.6 GB
INFO:ActiveLearning:All modules integrated successfully!


=== Integration Test ===
Final setup: seed=42, device=cuda, batch_size=8


In [6]:
from data.loader import load_mit_dataset, load_dataset_from_config
from data.transforms import get_ner_statistics, log_ner_statistics  # Stats moved here
from config.settings import settings
from utils.logging import setup_logging

print("=== Testing Data Loader ===")

# Setup logger for testing
logger = setup_logging(log_dir="../logs", logger_name="DataTest")

# Test with dummy data (since actual files might not exist)

# Try to load real data if available
test_data_path = settings.data_path / settings.test_file
labels_path = settings.data_path / settings.labels_file

if test_data_path.exists() and labels_path.exists():
    test_data, entity_types = load_mit_dataset(str(test_data_path), str(labels_path), "test")
    print(f"Loaded real data: {len(test_data)} examples, {len(entity_types)} entity types")
    
    # Test unified stats function (no duplication now)
    stats = get_ner_statistics(test_data, entity_types)
    print(f"Dataset stats: {stats}")
    
    # Log stats using unified function
    log_ner_statistics(test_data, "Train", logger, entity_types)


INFO:DataTest:================================================================================
INFO:DataTest:ACTIVE LEARNING PIPELINE WITH PROPER TRAIN/TEST SEPARATION
INFO:DataTest:================================================================================
INFO:DataTest:Log file: ../logs/active_learning_20250910_190512.log
INFO:DataTest:Train Dataset Statistics:
INFO:DataTest:  Total examples: 2442
INFO:DataTest:  Avg num tokens: 10.10
INFO:DataTest:  Avg num entities: 2.15
INFO:DataTest:  Total entities: 5243
INFO:DataTest:  Unique entity types: 12
INFO:DataTest:  Top entity types: [('genre', 1117), ('actor', 812), ('year', 720), ('title', 561), ('plot', 491)]


=== Testing Data Loader ===
Loading test data from: /opt/app/notebooks/abhishek/active_gliner/data/mit-movie/test.json
Processed 2442 examples
Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']
Loaded real data: 2442 examples, 12 entity types
Dataset stats: {'total_examples': 2442, 'avg_num_tokens': 10.104422604422604, 'avg_num_entities': 2.147010647010647, 'total_entities': 5243, 'unique_entity_types': 12, 'entity_type_counts': Counter({'genre': 1117, 'actor': 812, 'year': 720, 'title': 561, 'plot': 491, 'director': 456, 'average ratings': 449, 'rating': 408, 'character': 89, 'review': 56, 'song': 54, 'trailer': 30}), 'entity_type_coverage': {'genre': 1117, 'year': 720, 'plot': 491, 'average ratings': 449, 'actor': 812, 'title': 561, 'song': 54, 'character': 89, 'rating': 408, 'review': 56, 'director': 456, 'trailer': 30}}


In [7]:
from data.loader import load_json_file
from selection.strategies import get_lowest_score_examples_sorted

low_n=load_json_file("../results/low_score_1000_examples.json")


print(len(low_n))



1000


In [9]:
from generation.simple_generator import *


# Initialize generator
generator = SyntheticDataGenerator()

# Generate synthetic data
synthetic_data = generator.generate(
    corrected_examples=low_n,
    num_samples=1000,
    entity_types=entity_types,
    countries=["USA", "Canada", "UK","India","france"],
    genres=["action","comedy","serious","adventure","sports"],
    subject="movie reviews"
)


synthetic_data[0]

SYNTHETIC DATA GENERATION
Subject: movie reviews
Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']
Countries: ['USA', 'Canada', 'UK', 'India', 'france']
Genres: ['action', 'comedy', 'serious', 'adventure', 'sports']
Template examples: 1000
Target samples: 1000


Generating:   0%|          | 1/1000 [00:04<1:15:53,  4.56s/it]

Generated 1/1000 samples...


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"
Generating:   0%|          | 2/1000 [00:06<51:57,  3.12s/it]  

⚠️ JSON parsing failed for sample 2: Expecting ',' delimiter: line 25 column 20 (char 450)


...

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"
Generating:  99%|█████████▉| 991/1000 [1:05:34<00:26,  2.95s/it]

Generated 991/1000 samples...


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"
Generating: 100%|██████████| 1000/1000 [1:06:13<00:00,  3.97s/it]


✅ Successfully generated 898/1000 raw samples
Converting 898 synthetic examples to NER format...
Conversion completed: 898 examples, 0 errors
📝 Converted to NER format: 898 examples
🧹 Final cleaned examples: 898
📊 Average entities per example: 8.0
📈 Entity distribution: {'title': 1065, 'year': 853, 'rating': 376, 'review': 476, 'genre': 941, 'character': 649, 'actor': 1014, 'trailer': 231, 'director': 579, 'song': 322, 'average ratings': 367, 'plot': 300}


{'tokenized_text': ['The',
  'latest',
  'action',
  'movie',
  ',',
  "'",
  'Fast',
  '&',
  'Furious',
  'Presents',
  ':',
  'Hobbs',
  '&',
  'Shaw',
  "'",
  ',',
  'released',
  'in',
  '2019',
  ',',
  'has',
  'received',
  'a',
  'rating',
  'of',
  '7',
  '.',
  '5',
  'from',
  'IMDb',
  '.',
  'The',
  'plot',
  'revolves',
  'around',
  'Luke',
  'Hobbs',
  'and',
  'Deckard',
  'Shaw',
  'joining',
  'forces',
  'to',
  'stop',
  'a',
  'cyber-genetically',
  'enhanced',
  'villain',
  '.',
  'Dwayne',
  'Johnson',
  'as',
  'Hobbs',
  'and',
  'Jason',
  'Statham',
  'as',
  'Shaw',
  'are',
  'the',
  'main',
  'actors',
  'in',
  'this',
  'high-octane',
  'action',
  'film',
  '.',
  'The',
  'trailer',
  'for',
  "'",
  'Fast',
  '&',
  'Furious',
  'Presents',
  ':',
  'Hobbs',
  '&',
  'Shaw',
  "'",
  'was',
  'released',
  'on',
  'YouTube',
  ',',
  'providing',
  'an',
  'exciting',
  'sneak',
  'peek',
  'into',
  'the',
  'movie',
  '.'],
 'ner': [[6, 13, 't

In [10]:
from data.loader import save_json_file


save_json_file(synthetic_data,'../results/syn_1000_examples.json')

Saved data to ../results/syn_1000_examples.json


: 

: 